In [0]:
# 1. Parámetros dinámicos puros
dbutils.widgets.text("env", "dev", "Ambiente")
dbutils.widgets.text("carpeta", "catalogos", "Carpeta origen")
dbutils.widgets.text("tabla", "proveedores", "Nombre tabla")

ambiente = dbutils.widgets.get("env")
carpeta_origen = dbutils.widgets.get("carpeta")
nombre_tabla = dbutils.widgets.get("tabla")

# 2. Rutas
storage_account = "adlsferreteria26"
container = "landing-zone"
ruta_origen = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{carpeta_origen}/"
ruta_checkpoint = f"abfss://{container}@{storage_account}.dfs.core.windows.net/_checkpoints/{ambiente}/bronze/{nombre_tabla}"
tabla_destino = f"ferreteria_{ambiente}.bronze.{nombre_tabla}"

# 3. Lectura y Escritura (Sin ciclos, directo a la acción)
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", ruta_checkpoint)
    .option("pathGlobFilter", f"{nombre_tabla}*.csv")
    .option("header", "true")
    .load(ruta_origen)
)

(df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", ruta_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(tabla_destino)
)